In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


import sys
sys.path.append("../utils")
from plotting_utils import format_top_3, plot_metric_grouped_by, plot_bio_vs_batch_correction

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
base_path = ".."
mali_path = "../../MALI"
scratch_path = ".."


# save_name = f"real_batches_no_masking" #dataset}".format(dataset = dataset_name)
save_name = f"real_batches_investigation_metrics" 
# save_name = f"real_batches_no_pca_no_hvg" 
# save_name = f"real_batches_with_pca_no_hvg" 
save_path = f"{scratch_path}/results/{save_name}"


create a new summary df based on aggregating the dfs in the n_comp subfolders (only needs to be ran once)

In [ ]:
batches = ["1","2","3","4","5","6", "A1", "A2", "A3", "A4", "A5", "A6", "B1", "B2", "B3", "B4"]

all_results_df = pd.DataFrame()
for i1, batch1 in enumerate(batches):
    save_path_subfolder = f"{save_path}/{batch1}"
    try:
        df = pd.read_csv(f"{save_path_subfolder}/real_batches_results.csv")
    except FileNotFoundError:
        df = pd.DataFrame()
        print(f"File not found: {save_path_subfolder}/real_batches_results.csv")

    df["batch"] = batch1
    all_results_df = pd.concat([all_results_df, df], ignore_index=True)
            
all_results_df    
all_results_df.to_csv(f"{save_path}/real_batches_results.csv", index=False)
                

In [ ]:
all_results_df.batch


In [ ]:
all_results_df

# Real batches data

In [ ]:
# exp = "real_batches_20pct_masking"
# exp = "real_batches_no_masking"
exp=save_name
results_path = f"../results/{exp}"

# load results
results_df = pd.read_csv(f"{results_path}/real_batches_results.csv")


metric_type = results_df.loc[results_df["method"] == "Metric Type"]
metric_type = metric_type.drop(columns=['method']).drop_duplicates()
results_df = results_df.loc[results_df["method"] != "Metric Type"]


# # convert relevant columns to numeric
num_cols = results_df.columns.difference(['method', 'batch'])
results_df[num_cols] = results_df[num_cols].apply(pd.to_numeric, errors='coerce')


# results_df = results_df.drop(0, axis = 0) # remove one run from debugging
# results_df.drop_duplicates(subset=['method'], keep='last', inplace=True)


# delete rows where n_components is NA
# results_df = results_df.loc[~results_df["n_components"].isna()]


# remove n_components in the model column (it was just used to not overwrite adata, and it's already another column)
results_df["method"] = results_df["method"].str.replace(r'_\d+_components', '', regex=True)

In [ ]:
# rename FOSTA PHATE t2 to FoSTA
results_df["method"] = results_df["method"].replace({"FoSTA_PHATE_t2": "FoSTA"})
results_df["method"] = results_df["method"].replace({"RFMALI_PHATE_t2": "RFMALI"})

# methods_to_remove= ["RFMALI", "LIGER"]
# results_df = results_df[~results_df["method"].isin(methods_to_remove)]

In [ ]:
# results_df["Bio conservation"]
results_df.columns

In [ ]:
sns.barplot(data=results_df, x="batch", y="Bio conservation", hue="method")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
sns.barplot(data=results_df, x="batch", y="cell_cycle_score", hue="method")

# make plots

In [ ]:
plot_metric_grouped_by(results_df, groupby_cols=["method"])

In [ ]:
plot_metric_grouped_by(results_df, groupby_cols=["batch1_batch2", "method"], annotate=False, fig_len_factor=4)

In [ ]:
results_df["batch1_batch2"] = results_df["batch1"] + "_" + results_df["batch2"]

In [ ]:
len(results_df["batch1_batch2"].unique())

In [ ]:
plot_bio_vs_batch_correction(results_df, title= "Real batches")

# look for the best performance

In [ ]:
best_idx_total = results_df.loc[results_df["method"] == "FoSTA", "Total"].idxmax()
best_idx_bio = results_df.loc[results_df["method"] == "FoSTA", "Bio conservation"].idxmax()
best_idx_batch = results_df.loc[results_df["method"] == "FoSTA", "Batch correction"].idxmax()

best_pair_total = results_df.loc[best_idx_total]["batch1_batch2"]
best_pair_bio = results_df.loc[best_idx_bio]["batch1_batch2"]
best_pair_batch = results_df.loc[best_idx_batch]["batch1_batch2"]

total_pair_df = results_df.loc[results_df["batch1_batch2"] == best_pair_total]
bio_pair_df = results_df.loc[results_df["batch1_batch2"] == best_pair_bio]
batch_pair_df = results_df.loc[results_df["batch1_batch2"] == best_pair_batch]

In [ ]:
plot_bio_vs_batch_correction(total_pair_df, title = best_pair_total)

In [ ]:
plot_bio_vs_batch_correction(bio_pair_df, title = best_pair_bio)

In [ ]:
plot_bio_vs_batch_correction(batch_pair_df, title = best_pair_batch)

# only consider batches within the same set

In [ ]:
results_df["batch1"] = results_df["batch1"].astype(str)
results_df["batch2"] = results_df["batch2"].astype(str)

In [ ]:
def get_set_of_batch(batch):
    if batch.startswith("A"):
        return "A"
    elif batch.startswith("B"):
        return "B"
    else:
        return "0"
    
results_df["same_batch_set"] = (results_df["batch1"].apply(get_set_of_batch) == results_df["batch2"].apply(get_set_of_batch))

In [ ]:
same_batch_results_df = results_df.loc[results_df["same_batch_set"] == True]

In [ ]:
plot_bio_vs_batch_correction(same_batch_results_df)